In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp -r "/content/drive/MyDrive/Vietnamese Food Prediction" "/content/"

In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Rescaling, RandomFlip, RandomRotation, RandomZoom, Conv2D, BatchNormalization, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt

In [4]:
train_dir = "/content/Vietnamese Food Prediction/Train"
validation_dir = "/content/Vietnamese Food Prediction/Validate"
test_dir = "/content/Vietnamese Food Prediction/Test"
img_width, img_height = 128, 128
batch_size = 32

In [5]:
#Tải dữ liệu
train_generator = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=(img_width, img_height),
    batch_size=batch_size)

validation_generator = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=(img_width, img_height),
    batch_size=batch_size)

test_generator = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=(img_width, img_height),
    batch_size=batch_size,
    shuffle=False)

Found 6699 files belonging to 10 classes.
Found 956 files belonging to 10 classes.
Found 1920 files belonging to 10 classes.


In [7]:
AUTOTUNE = tf.data.AUTOTUNE
train_generator = train_generator.cache().prefetch(buffer_size=AUTOTUNE)
validation_generator = validation_generator.cache().prefetch(buffer_size=AUTOTUNE)

In [8]:
model = Sequential([
    Rescaling(1./255, input_shape=(img_width, img_height, 3)),
    #Tăng cường dữ liệu
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1),

    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(), #kéo dữ liệu về ổn định để model dễ học hơn
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [9]:
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,946 (437.29 KB)

 Trainable params: 111,498 (435.54 KB)

 Non-trainable params: 448 (1.75 KB)

In [10]:
#Huấn luyện mô hình CNN
epochs = 40
model_checkpoint = ModelCheckpoint('/content/drive/MyDrive/best_food_model.h5', monitor='val_accuracy', save_best_only=True)
history = model.fit(train_generator, validation_data=validation_generator, epochs=epochs, callbacks=model_checkpoint)

Epoch 1/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.2538 - loss: 2.1546

210/210 ━━━━━━━━━━━━━━━━━━━━ 77s 315ms/step - accuracy: 0.3109 - loss: 1.9910 - val_accuracy: 0.2207 - val_loss: 2.3045
Epoch 2/40
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3971 - loss: 1.7475

210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.4156 - loss: 1.6972 - val_accuracy: 0.3086 - val_loss: 2.0939
Epoch 3/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.4748 - loss: 1.5205 - val_accuracy: 0.2605 - val_loss: 2.5541
Epoch 4/40
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5082 - loss: 1.4219

210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.5260 - loss: 1.3847 - val_accuracy: 0.3598 - val_loss: 1.8608
Epoch 5/40
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.5664 - loss: 1.2828

210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.5756 - loss: 1.2631 - val_accuracy: 0.4351 - val_loss: 1.5813
Epoch 6/40
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5906 - loss: 1.2153

210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.6049 - loss: 1.1795 - val_accuracy: 0.4550 - val_loss: 1.6176
Epoch 7/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.6346 - loss: 1.1012 - val_accuracy: 0.3912 - val_loss: 2.2252
Epoch 8/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.6543 - loss: 1.0360 - val_accuracy: 0.3828 - val_loss: 1.8702
Epoch 9/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accuracy: 0.6741 - loss: 0.9881 - val_accuracy: 0.3870 - val_loss: 2.2947
Epoch 10/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.6913 - loss: 0.9370 - val_accuracy: 0.3923 - val_loss: 2.2341
Epoch 11/40
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7080 - loss: 0.9017

210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.7123 - loss: 0.8879 - val_accuracy: 0.5199 - val_loss: 1.6087
Epoch 12/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.7212 - loss: 0.8610 - val_accuracy: 0.4759 - val_loss: 1.7729
Epoch 13/40
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7177 - loss: 0.8460

210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - accuracy: 0.7237 - loss: 0.8349 - val_accuracy: 0.5282 - val_loss: 1.6044
Epoch 14/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.7379 - loss: 0.8036 - val_accuracy: 0.4864 - val_loss: 1.6661
Epoch 15/40
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7499 - loss: 0.7683

210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.7510 - loss: 0.7661 - val_accuracy: 0.5983 - val_loss: 1.2268
Epoch 16/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 10s 42ms/step - accuracy: 0.7577 - loss: 0.7498 - val_accuracy: 0.5293 - val_loss: 1.6494
Epoch 17/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.7576 - loss: 0.7443 - val_accuracy: 0.5941 - val_loss: 1.3355
Epoch 18/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.7680 - loss: 0.7030 - val_accuracy: 0.5471 - val_loss: 1.4787
Epoch 19/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.7676 - loss: 0.7019 - val_accuracy: 0.4770 - val_loss: 1.9969
Epoch 20/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.7827 - loss: 0.6685 - val_accuracy: 0.5178 - val_loss: 1.6519
Epoch 21/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.7871 - loss: 0.6529 - val_accuracy: 0.5805 - val_loss: 1.5060
Epoch 22/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.7844 - loss: 0.6543 - val_accurac

210/210 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - accuracy: 0.8004 - loss: 0.6010 - val_accuracy: 0.6851 - val_loss: 1.1202
Epoch 26/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accuracy: 0.8003 - loss: 0.5902 - val_accuracy: 0.4634 - val_loss: 2.3297
Epoch 27/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.8086 - loss: 0.5759 - val_accuracy: 0.4467 - val_loss: 2.6139
Epoch 28/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.8144 - loss: 0.5681 - val_accuracy: 0.5042 - val_loss: 2.1038
Epoch 29/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.8194 - loss: 0.5422 - val_accuracy: 0.4895 - val_loss: 1.8314
Epoch 30/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.8207 - loss: 0.5476 - val_accuracy: 0.3912 - val_loss: 3.6344
Epoch 31/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.8268 - loss: 0.5317 - val_accuracy: 0.5031 - val_loss: 2.0017
Epoch 32/40
210/210 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.8221 - loss: 0.5237 - val_accurac

In [11]:
#Đánh giá kết quả mô hình
best_model = tf.keras.models.load_model('/content/drive/MyDrive/best_food_model.h5')
train_loss, train_accuracy = best_model.evaluate(train_generator)
val_loss, val_accuracy = best_model.evaluate(validation_generator)
test_loss, test_accuracy = best_model.evaluate(test_generator)
print(f"Độ chính xác thực tế trên tập Train: {train_accuracy*100:.2f}%")
print(f"Độ chính xác thực tế trên tập Validation: {val_accuracy*100:.2f}%")
print(f"Độ chính xác thực tế trên tập Test: {test_accuracy*100:.2f}%")

210/210 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.7323 - loss: 0.8207
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6851 - loss: 1.1202
60/60 ━━━━━━━━━━━━━━━━━━━━ 14s 226ms/step - accuracy: 0.6818 - loss: 1.0722
Độ chính xác thực tế trên tập Train: 73.23%
Độ chính xác thực tế trên tập Validation: 68.51%
Độ chính xác thực tế trên tập Test: 68.18%
